# L1. The On-Device Memory Problem

Your AI has a model on the device and it can search vectors, but it has nowhere to **remember** anything. In this short notebook you set up the fix: a **Qdrant Edge** memory store that lives inside your process, on your disk. The full build starts in L2.

## 1. API keys, the course way

In [ ]:
from dotenv import load_dotenv

load_dotenv()  # reads .env into environment variables

## 2. Configure the memory store

In [2]:
from qdrant_edge import EdgeConfig, EdgeVectorParams, Distance
from helper import NOMIC_DIM

config = EdgeConfig(vectors={
    "text": EdgeVectorParams(
        size=NOMIC_DIM, distance=Distance.Cosine
    )
})
print(f"One named vector: text, {NOMIC_DIM}-d, cosine distance")

One named vector: text, 768-d, cosine distance


## 3. Create it

In [3]:
from pathlib import Path
from qdrant_edge import EdgeShard

SHARD_DIR = "./first_shard"
Path(SHARD_DIR).mkdir(parents=True, exist_ok=True)
shard = EdgeShard.create(SHARD_DIR, config)
print("EdgeShard created at", SHARD_DIR)

EdgeShard created at ./first_shard


## 4. Write the first memory

In [4]:
from qdrant_edge import Point, UpdateOperation
from helper import embed_text

note = ("Great little coffee place on 5th with outdoor "
        "seating and fast wifi")
vec = embed_text([note])[0]

shard.update(UpdateOperation.upsert_points([
    Point(id=0, vector={"text": vec}, payload={"note": note})
]))
shard.optimize()
print("Memories stored:", shard.info().points_count)

Memories stored: 1


## 5. Money output: a database that is just files

In [5]:
shard.close()

files = [p for p in Path(SHARD_DIR).rglob("*") if p.is_file()]
for p in sorted(files)[:8]:
    print("  ", p.relative_to(SHARD_DIR))
print(f"\n{len(files)} plain files in a local folder — "
      "your AI's memory, no server")

   edge_config.json
   segments/a1ac9a62-3943-432d-bd21-c58ade57175f/mutable_id_tracker.mappings
   segments/a1ac9a62-3943-432d-bd21-c58ade57175f/mutable_id_tracker.versions
   segments/a1ac9a62-3943-432d-bd21-c58ade57175f/payload_index/config.json
   segments/a1ac9a62-3943-432d-bd21-c58ade57175f/payload_storage/bitmask.dat
   segments/a1ac9a62-3943-432d-bd21-c58ade57175f/payload_storage/config.json
   segments/a1ac9a62-3943-432d-bd21-c58ade57175f/payload_storage/gaps.dat
   segments/a1ac9a62-3943-432d-bd21-c58ade57175f/payload_storage/page_0.dat

18 plain files in a local folder — your AI's memory, no server


In [6]:
# Cleanup so the notebook re-runs cleanly. L2 builds its own store.
import shutil
shutil.rmtree(SHARD_DIR, ignore_errors=True)
print("Cleaned up", SHARD_DIR)

Cleaned up ./first_shard
